# Extractory — Colab GPU (Option 1: Cursor + Colab)

**In Cursor:** run `scripts/create-colab-upload.ps1`, then `chroma run`, then `npm run dev` after you paste the URL into `.env`.

**Here in Colab:** Runtime → **T4 GPU** → Run all cells.

See [COLAB_SETUP.md](./COLAB_SETUP.md) for the full guide.

In [ ]:
!nvidia-smi

In [ ]:
# Upload colab-upload.zip (create it in Cursor: .\scripts\create-colab-upload.ps1)
from google.colab import files
import os, glob, zipfile

print("Select colab-upload.zip from your PC...")
uploaded = files.upload()

zip_name = next((n for n in uploaded if n.endswith(".zip")), None)
if not zip_name:
    raise RuntimeError("Please upload colab-upload.zip")

with zipfile.ZipFile(zip_name, "r") as zf:
    zf.extractall("/content")

print("Extracted to /content:")
!find /content -name ocr_server.py 2>/dev/null

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os, sys

PROJECT_ROOT = None
for root, dirs, files in os.walk("/content"):
    if "ocr_server.py" in files and os.path.basename(root) == "colab":
        PROJECT_ROOT = os.path.dirname(root)
        break

if not PROJECT_ROOT:
    raise RuntimeError("colab/ocr_server.py not found. Re-run the upload cell with colab-upload.zip")

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)
!pip install -q -r colab/requirements.txt

In [ ]:
import subprocess, time

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

!ollama pull moondream
!ollama pull nomic-embed-text
!ollama pull llama3.2:3b
!ollama list

In [ ]:
import os
os.environ["COLAB_OCR_SECRET"] = ""  # optional — match COLAB_OCR_SECRET in Cursor .env
os.environ["OLLAMA_MODEL"] = "moondream"
os.environ["RAG_EMBEDDING_MODEL"] = "nomic-embed-text"
os.environ["RAG_CHAT_MODEL"] = "llama3.2:3b"
os.environ["PORT"] = "8080"

In [ ]:
# Paste ngrok token: https://dashboard.ngrok.com/get-started/your-authtoken
from pyngrok import ngrok
import threading
import uvicorn

NGROK_AUTHTOKEN = ""  # required for stable tunnels
if not NGROK_AUTHTOKEN:
    print("WARNING: Set NGROK_AUTHTOKEN above, then re-run this cell.")
else:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)

ngrok.kill()
tunnel = ngrok.connect(8080, bind_tls=True)
public_url = tunnel.public_url

print("=" * 60)
print("Paste this into COLAB_OCR_URL in Cursor .env:")
print(public_url)
print("Health:", f"{public_url}/health")
print("=" * 60)

def run_server():
    uvicorn.run("colab.ocr_server:app", host="0.0.0.0", port=8080, log_level="info")

threading.Thread(target=run_server, daemon=True).start()
print("Server running. Keep this Colab tab open while using the Cursor app.")

In [ ]:
import httpx
url = str(public_url).rstrip("/")
print(httpx.get(f"{url}/health", timeout=15.0).json())